# FIT4013 - Nhập môn Học máy | Nhóm 6 - Lớp CNTT 18-10
## Đề tài 6: Nhận dạng biển báo giao thông Đức (GTSRB - 43 lớp)
- **Nhóm thực hiện:** Lê Trọng Chiến (1871020089), Đặng Việt Anh (1871020683), Nguyễn Hữu Dũng (1871020172), Kiều Văn Nam (1871020416)
- **Chạy tái lập:** `Runtime -> Run all` trên Google Colab (tự động `git clone`, cài thư viện `requirements.txt`, tải GTSRB từ ERDA và kiểm thử Flask Web App).


### Buoc 1 - Dong bo ma nguon GitHub, cai dat thu vien va tai du lieu GTSRB


In [1]:
# [Cell 1] Dong bo ma nguon, cai dat moi truong (pinned requirements) & tai du lieu GTSRB (51.839 anh)
import os, sys, time
if not os.path.exists("src/pipeline.py"):
    if not os.path.exists("FIT4013-CNTT-1810-NHOM-6"):
        !git clone -q https://github.com/Chieenslee/FIT4013-CNTT-1810-NHOM-6.git
    %cd FIT4013-CNTT-1810-NHOM-6

!pip install -q -r requirements.txt

import numpy as np
import pandas as pd
import sklearn, cv2, skimage

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.data_loader import download_gtsrb_dataset, load_rgb_image
from src.constants import CLASS_NAMES_VI, CLASS_SHORT_VI
from src.features import crop_center_margin
from src.pipeline import (
    RANDOM_STATE, _wilson_ci_pct, apply_holm_bonferroni, bootstrap_ci,
    compute_metrics, count_unique_tracks, extract_track_ids, fit_eval_model,
    fit_logistic_error_model, load_all_images_and_features, mcnemar_test,
    plot_ablation_and_loss, plot_confusion_matrix_fig, plot_dt_hyperparams,
    plot_error_analysis_samples, plot_error_diagnostics, plot_knn_hyperparams,
    plot_model_comparison, plot_pca_analysis, prepare_scaled_split,
)

DATA_ROOT = os.path.join(os.getcwd(), "data")
train_csv, test_csv = download_gtsrb_dataset(DATA_ROOT)
df_train_full = pd.read_csv(train_csv)
df_test = pd.read_csv(test_csv)
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | scikit-learn {sklearn.__version__} | OpenCV {cv2.__version__}")
print(f"Loaded CSVs -> Train: {len(df_train_full):,} rows | Test: {len(df_test):,} rows | Classes: {df_train_full['ClassId'].nunique()}")

Python 3.13.7 | NumPy 2.4.4 | scikit-learn 1.8.0 | OpenCV 4.13.0
Loaded CSVs -> Train: 39,209 rows | Test: 12,630 rows | Classes: 43

### Buoc 2 - Kham pha du lieu (EDA) va kiem tra track vat ly


In [2]:
# [Cell 2] Phan tich phan bo lop, track vat ly, kich thuoc anh va chat luong du lieu
track_stats = count_unique_tracks(df_train_full)
cls_counts = df_train_full["ClassId"].value_counts()
test_cls_counts = df_test["ClassId"].value_counts()
missing_cnt = int(df_train_full.isna().sum().sum() + df_test.isna().sum().sum())
dup_cnt = int(df_train_full.duplicated().sum() + df_test.duplicated().sum())

print("=== KET QUA EDA TAP DU LIEU GTSRB ===")
print(f"Tong anh Train: {len(df_train_full):,} | Test: {len(df_test):,} | Tong cong: {len(df_train_full)+len(df_test):,}")
print(f"So track vat ly (Train): {track_stats['unique_tracks']:,} track | Phan bo anh/track: {track_stats['frames_per_track_counts']}")
print(f"Phan bo lop Train: min={cls_counts.min()} (lop {cls_counts.idxmin()}), max={cls_counts.max()} (lop {cls_counts.idxmax()}), ratio={cls_counts.max()/cls_counts.min():.2f}x")
print(f"Phan bo lop Test : min={test_cls_counts.min()} anh, max={test_cls_counts.max()} anh")
print(f"Kich thuoc anh (Train): Width {df_train_full['Width'].min()}..{df_train_full['Width'].max()}px (median {df_train_full['Width'].median():.0f}px) | Height {df_train_full['Height'].min()}..{df_train_full['Height'].max()}px")
print(f"Thieu du lieu (NaN): {missing_cnt} | Ban ghi trung lap: {dup_cnt}")

=== KET QUA EDA TAP DU LIEU GTSRB ===
Tong anh Train: 39,209 | Test: 12,630 | Tong cong: 51,839
So track vat ly (Train): 1,307 track | Phan bo anh/track: {29: 1, 30: 1306}
Phan bo lop Train: min=210 (lop 0), max=2250 (lop 2), ratio=10.71x
Phan bo lop Test : min=60 anh, max=750 anh
Kich thuoc anh (Train): Width 25..243px (median 43px) | Height 25..225px
Thieu du lieu (NaN): 0 | Ban ghi trung lap: 0

### Buoc 3 - Trich xuat dac trung F0..F5 (gom F3_noCLAHE) va chia StratifiedGroupKFold


In [3]:
# [Cell 3] Trich xuat dac trung va chia Train/GroupVal tach track hoan toan (0 track trung lap)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier

CACHE_DIR = os.path.join(os.getcwd(), "cache")
os.makedirs(CACHE_DIR, exist_ok=True)
feats_tr_full = load_all_images_and_features(
    df_train_full, DATA_ROOT, cache_path=os.path.join(CACHE_DIR, "cache_feats_TRAIN_FULL.npz")
)
feats_te = load_all_images_and_features(
    df_test, DATA_ROOT, cache_path=os.path.join(CACHE_DIR, "cache_feats_TEST.npz")
)

track_ids = extract_track_ids(df_train_full)
sgkf = StratifiedGroupKFold(n_splits=5)
idx_tr, idx_val = next(sgkf.split(np.zeros(len(df_train_full)), feats_tr_full["y"], groups=track_ids))
y_tr, y_val, y_te = feats_tr_full["y"][idx_tr], feats_tr_full["y"][idx_val], feats_te["y"]
overlap_tracks = len(set(track_ids[idx_tr]).intersection(set(track_ids[idx_val])))

print(f"StratifiedGroupKFold(5) -> Train={len(idx_tr):,} ({len(np.unique(track_ids[idx_tr]))} tracks) | GroupVal={len(idx_val):,} ({len(np.unique(track_ids[idx_val]))} tracks) | Shared tracks={overlap_tracks}")
print(f"Ti le dien tich ROI trung binh: {feats_tr_full['roi_ratio'].mean()*100:.2f}% | Do sang Train={feats_tr_full['brightness'].mean():.2f} +/- {feats_tr_full['brightness'].std():.2f}, Test={feats_te['brightness'].mean():.2f}")

r_tr, r_val = train_test_split(np.arange(len(df_train_full)), test_size=0.2, random_state=RANDOM_STATE, stratify=feats_tr_full["y"])
xr_tr, xr_val, xr_te, _, _, _ = prepare_scaled_split(feats_tr_full["F4"][r_tr], feats_tr_full["F4"][r_val], feats_te["F4"], pca_dim=128)
_, r_val_m, r_te_m, _, _ = fit_eval_model(KNeighborsClassifier(n_neighbors=5, weights="distance", algorithm="brute", n_jobs=-1), xr_tr, feats_tr_full["y"][r_tr], xr_val, feats_tr_full["y"][r_val], xr_te, y_te)
print(f"Doi chung ro ri track -> Random Frame Val={r_val_m['accuracy']*100:.2f}% (Test={r_te_m['accuracy']*100:.2f}%) vs. Track-disjoint GroupVal")

StratifiedGroupKFold(5) -> Train=31,349 (1045 tracks) | GroupVal=7,860 (262 tracks) | Shared tracks=0
Ti le dien tich ROI trung binh: 56.19% | Do sang Train=82.01 +/- 47.64, Test=81.38
Doi chung ro ri track -> Random Frame Val=98.66% (Test=89.75%) vs. Track-disjoint GroupVal=91.42% (Test=89.18%)

### Buoc 4 - Thi nghiem 1 (9 cau hinh F0..F5 + F3_noCLAHE) & Kiem chung trong so mau alpha


In [4]:
# [Cell 4] Exp 1 (9 tang dac trung) + Ablation trong so mau alpha in {0, 0.25, 0.5, 0.75, 1.0}
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

feature_specs = [
    ("F0: No-ROI Raw Grayscale (1024)", "F0", None),
    ("F1: ROI Raw Grayscale - Baseline (1024)", "F1", None),
    ("F2: ROI Raw RGB (3072)", "F2", None),
    ("F3_noCLAHE: ROI + HOG Khong CLAHE (1764)", "F3_noCLAHE", None),
    ("F3: ROI + CLAHE + HOG (1764)", "F3", None),
    ("F3_PCA: ROI + CLAHE + HOG + PCA (128)", "F3", 128),
    ("F4_noCLAHE: ROI + HOG + Color (1860)", "F4_noCLAHE", None),
    ("F4: ROI + CLAHE + HOG + Color (1860)", "F4", None),
    ("F5: ROI + CLAHE + HOG + Color + PCA (128)", "F4", 128),
]

exp1_results, exp1_preds, cache_splits = [], {}, {}
t_exp1_start = time.perf_counter()
print(f"{'Dac trung':<45} | {'Chieu':>5} | {'KNN GV':>7} | {'KNN Te':>7} | {'DT Te':>7} | {'GNB Te':>7}")
print("-" * 96)
for label, src_key, pca_dim in feature_specs:
    split_key = label.split(":")[0]
    X_tr_s, X_val_s, X_te_s, sc, pca_obj, exp_var = prepare_scaled_split(
        feats_tr_full[src_key][idx_tr], feats_tr_full[src_key][idx_val], feats_te[src_key], pca_dim=pca_dim
    )
    cache_splits[split_key] = (X_tr_s, X_val_s, X_te_s, sc, pca_obj, exp_var)
    _, k_val, k_te, _, k_p = fit_eval_model(KNeighborsClassifier(n_neighbors=5, weights="distance", algorithm="brute", n_jobs=-1), X_tr_s, y_tr, X_val_s, y_val, X_te_s, y_te)
    _, d_val, d_te, _, d_p = fit_eval_model(DecisionTreeClassifier(criterion="gini", max_depth=20, random_state=RANDOM_STATE), X_tr_s, y_tr, X_val_s, y_val, X_te_s, y_te)
    _, g_val, g_te, _, g_p = fit_eval_model(GaussianNB(), X_tr_s, y_tr, X_val_s, y_val, X_te_s, y_te)
    exp1_preds[f"knn_{split_key}"], exp1_preds[f"dt_{split_key}"], exp1_preds[f"gnb_{split_key}"] = k_p, d_p, g_p
    exp1_results.append({"feature": label, "dims": X_tr_s.shape[1], "explained_variance": exp_var, "knn_val": k_val, "knn_test": k_te, "dt_val": d_val, "dt_test": d_te, "gnb_val": g_val, "gnb_test": g_te})
    print(f"{label:<45} | {X_tr_s.shape[1]:>5} | {k_val['accuracy']*100:>6.2f}% | {k_te['accuracy']*100:>6.2f}% | {d_te['accuracy']*100:>6.2f}% | {g_te['accuracy']*100:>6.2f}%")

X_tr_f5, X_val_f5, X_te_f5, scaler_f5, pca_f5, _ = cache_splits["F5"]
X_tr_f3p, X_val_f3p, X_te_f3p, scaler_f3p, pca_f3p, _ = cache_splits["F3_PCA"]
X_tr_f4, X_val_f4, X_te_f4, _, _, _ = cache_splits["F4"]
X_tr_f1, X_val_f1, X_te_f1, _, _, _ = cache_splits["F1"]

color_weight_ablation = []
print("\n=== KIEM CHUNG TRONG SO KHOI MAU ALPHA (HOG 1764d + alpha * Color 96d) ===")
for alpha in [0.0, 0.25, 0.50, 0.75, 1.0]:
    Xw_tr, Xw_val, Xw_te = X_tr_f4.copy(), X_val_f4.copy(), X_te_f4.copy()
    Xw_tr[:, 1764:] *= alpha; Xw_val[:, 1764:] *= alpha; Xw_te[:, 1764:] *= alpha
    _, mv_r, mte_r, _, _ = fit_eval_model(KNeighborsClassifier(n_neighbors=11, weights="distance", algorithm="brute", n_jobs=-1), Xw_tr, y_tr, Xw_val, y_val, Xw_te, y_te)
    pca_a = PCA(n_components=128, svd_solver="full")
    _, mv_p, mte_p, _, _ = fit_eval_model(KNeighborsClassifier(n_neighbors=11, weights="distance", algorithm="brute", n_jobs=-1), pca_a.fit_transform(Xw_tr).astype(np.float32), y_tr, pca_a.transform(Xw_val).astype(np.float32), y_val, pca_a.transform(Xw_te).astype(np.float32), y_te)
    color_weight_ablation.append({"alpha": alpha, "gv_raw1860_acc": mv_r["accuracy"], "te_raw1860_acc": mte_r["accuracy"], "gv_pca128_acc": mv_p["accuracy"], "te_pca128_acc": mte_p["accuracy"]})
    print(f"alpha={alpha:4.2f} | Raw1860 GV={mv_r['accuracy']*100:5.2f}% Te={mte_r['accuracy']*100:5.2f}% | PCA128 GV={mv_p['accuracy']*100:5.2f}% Te={mte_p['accuracy']*100:5.2f}%")
print(f"-> Tong thoi gian Exp 1 + Color Weight Ablation: {time.perf_counter()-t_exp1_start:.1f}s")

Dac trung                                     | Chieu |  KNN GV |  KNN Te |   DT Te |  GNB Te
------------------------------------------------------------------------------------------------
F0: No-ROI Raw Grayscale (1024)               |  1024 |  33.98% |  33.55% |  50.22% |   9.51%
F1: ROI Raw Grayscale - Baseline (1024)       |  1024 |  50.84% |  50.59% |  56.48% |  14.18%
F2: ROI Raw RGB (3072)                        |  3072 |  52.99% |  52.89% |  57.55% |  17.32%
F3: ROI + CLAHE + HOG (1764)                  |  1764 |  92.30% |  89.86% |  72.05% |  81.01%
F3_PCA: ROI + CLAHE + HOG + PCA (128)         |   128 |  92.06% |  90.02% |  68.58% |  88.37%
F4_noCLAHE: ROI + HOG + Color (1860)          |  1860 |  91.50% |  88.42% |  73.87% |  80.32%
F4: ROI + CLAHE + HOG + Color (1860)          |  1860 |  91.92% |  89.34% |  72.32% |  81.37%
F5: ROI + CLAHE + HOG + Color + PCA (128)     |   128 |  91.42% |  89.18% |  68.04% |  88.20%

### Buoc 5 - Thi nghiem 2 & 3: Quet luoi K-NN mo rong (k=1..31, 48 cau hinh) & DT tren GroupVal


In [5]:
# [Cell 5] Exp 2 (48 cau hinh K-NN k=1..31 tren F5 & F3_PCA) + Exp 3 (12 cau hinh DT) — STRICTLY ON GroupVal
from sklearn.metrics import accuracy_score

exp2_results = []
print("=== EXP 2: LUOI 48 CAU HINH K-NN k in {1,3,5,7,11,15,21,31} TREN F5 (GROUP_VAL) ===")
for k in [1, 3, 5, 7, 11, 15, 21, 31]:
    for weights in ["uniform", "distance"]:
        for metric in ["euclidean", "manhattan", "cosine"]:
            clf = KNeighborsClassifier(n_neighbors=k, weights=weights, metric=metric, algorithm="brute", n_jobs=-1)
            fk, m_val, _, _, _ = fit_eval_model(clf, X_tr_f5, y_tr, X_val_f5, y_val)
            tr_acc = 1.0 if weights == "distance" else float(accuracy_score(y_tr, fk.predict(X_tr_f5)))
            exp2_results.append({"k": k, "weights": weights, "metric": metric, "train_accuracy": tr_acc, "group_val": m_val})

for r in sorted(exp2_results, key=lambda x: x["group_val"]["accuracy"], reverse=True)[:10]:
    mv = r["group_val"]
    print(f"k={r['k']:<2} | {r['weights']:<8} | {r['metric']:<9} | Train={r['train_accuracy']*100:6.2f}% | GroupVal={mv['accuracy']*100:6.2f}% | GV F1={mv['f1_macro']*100:6.2f}%")
best_knn_cfg = max(exp2_results, key=lambda r: r["group_val"]["accuracy"])

exp2_f3p_results = []
for k in [3, 5, 7, 11, 15, 21, 31]:
    for metric in ["euclidean", "manhattan", "cosine"]:
        _, mv_3, _, _, _ = fit_eval_model(KNeighborsClassifier(n_neighbors=k, weights="distance", metric=metric, algorithm="brute", n_jobs=-1), X_tr_f3p, y_tr, X_val_f3p, y_val)
        exp2_f3p_results.append({"k": k, "weights": "distance", "metric": metric, "group_val": mv_3})
best_f3p_cfg = max(exp2_f3p_results, key=lambda r: r["group_val"]["accuracy"])

exp3_results = []
print("\n=== EXP 3: 12 CAU HINH DECISION TREE TREN F5 (GROUP_VAL) ===")
for crit in ["gini", "entropy"]:
    for depth in [5, 10, 15, 20, 30, None]:
        dt, m_val, _, _, _ = fit_eval_model(DecisionTreeClassifier(criterion=crit, max_depth=depth, random_state=RANDOM_STATE), X_tr_f5, y_tr, X_val_f5, y_val)
        tr_acc = float(accuracy_score(y_tr, dt.predict(X_tr_f5)))
        exp3_results.append({"criterion": crit, "max_depth": depth, "train_accuracy": tr_acc, "group_val": m_val})
        print(f"crit={crit:<7} | depth={str(depth):<4} | Train={tr_acc*100:6.2f}% | GroupVal={m_val['accuracy']*100:6.2f}% | GV F1={m_val['f1_macro']*100:6.2f}%")

best_dt_cfg = max(exp3_results, key=lambda r: r["group_val"]["accuracy"])
print(f"-> Best K-NN F3-PCA: k={best_f3p_cfg['k']}, {best_f3p_cfg['metric']} -> GV={best_f3p_cfg['group_val']['accuracy']*100:.2f}%")
print(f"-> Best K-NN F5    : k={best_knn_cfg['k']}, {best_knn_cfg['weights']}, {best_knn_cfg['metric']} -> GV={best_knn_cfg['group_val']['accuracy']*100:.2f}%")
print(f"-> Best DT   F5    : crit={best_dt_cfg['criterion']}, depth={best_dt_cfg['max_depth']} -> GV={best_dt_cfg['group_val']['accuracy']*100:.2f}%")

=== EXP 2: LUOI 30 CAU HINH K-NN TREN F5 (GROUP_VAL — TOP 10) ===
k=11 | distance | euclidean | Train=100.00% | GroupVal= 92.24% | GV F1= 88.94%
k=11 | distance | manhattan | Train=100.00% | GroupVal= 92.19% | GV F1= 88.92%
k=11 | uniform  | euclidean | Train=100.00% | GroupVal= 92.09% | GV F1= 88.70%
k=11 | distance | cosine    | Train=100.00% | GroupVal= 91.95% | GV F1= 88.23%
k=7  | distance | manhattan | Train=100.00% | GroupVal= 91.87% | GV F1= 88.75%
k=11 | uniform  | manhattan | Train=100.00% | GroupVal= 91.86% | GV F1= 88.50%
k=7  | distance | euclidean | Train=100.00% | GroupVal= 91.81% | GV F1= 88.54%
k=11 | uniform  | cosine    | Train=100.00% | GroupVal= 91.65% | GV F1= 87.84%
k=7  | uniform  | manhattan | Train=100.00% | GroupVal= 91.63% | GV F1= 88.47%
k=7  | uniform  | euclidean | Train=100.00% | GroupVal= 91.49% | GV F1= 88.23%

=== EXP 3: 12 CAU HINH DECISION TREE TREN F5 (GROUP_VAL) ===
crit=gini    | depth=5    | Train= 29.14% | GroupVal= 29.99% | GV F1= 13.20%
crit=

### Buoc 6 - 5-Fold GroupKFold CV, Tinh chinh MLP/RF, Full-Train Refit & Test Evaluation


In [6]:
# [Cell 6] Exp 4 (PCA GroupVal), 5-fold GroupKFold CV (mean +/- std), MLP/RF tuning & Full-Train Refit
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

scaler_f4_s = StandardScaler()
X_tr_f4s = scaler_f4_s.fit_transform(feats_tr_full["F4"][idx_tr]).astype(np.float32)
X_val_f4s = scaler_f4_s.transform(feats_tr_full["F4"][idx_val]).astype(np.float32)
pca_512 = PCA(n_components=512, svd_solver="full")
X_tr_p512 = pca_512.fit_transform(X_tr_f4s).astype(np.float32)
X_val_p512 = pca_512.transform(X_val_f4s).astype(np.float32)
cum_var = np.cumsum(pca_512.explained_variance_ratio_)

exp4_results = []
for dim in [32, 64, 128, 256, 512, 1860]:
    Xd_tr, Xd_val, evr = (X_tr_f4s, X_val_f4s, 1.0) if dim == 1860 else (X_tr_p512[:, :dim], X_val_p512[:, :dim], float(cum_var[dim - 1]))
    _, m_val, _, _, _ = fit_eval_model(KNeighborsClassifier(n_neighbors=int(best_knn_cfg["k"]), weights="distance", metric="cosine", algorithm="brute", n_jobs=-1), Xd_tr, y_tr, Xd_val, y_val)
    exp4_results.append({"dims": dim, "explained_variance": evr, "group_val": m_val})

cv_buckets = {}
for ftr, fva in sgkf.split(np.zeros(len(df_train_full)), feats_tr_full["y"], groups=track_ids):
    yf_tr, yf_va = feats_tr_full["y"][ftr], feats_tr_full["y"][fva]
    Xf3_tr, Xf3_va, _, _, _, _ = prepare_scaled_split(feats_tr_full["F3"][ftr], feats_tr_full["F3"][fva], feats_te["F3"], pca_dim=128)
    Xf5_tr, Xf5_va, _, _, _, _ = prepare_scaled_split(feats_tr_full["F4"][ftr], feats_tr_full["F4"][fva], feats_te["F4"], pca_dim=128)
    for m_lbl, est, xm_t, xm_v in [
        ("K-NN F3_PCA (k=11)", KNeighborsClassifier(n_neighbors=11, weights="distance", metric="euclidean", algorithm="brute", n_jobs=-1), Xf3_tr, Xf3_va),
        ("K-NN F5 (k=11)", KNeighborsClassifier(n_neighbors=11, weights="distance", metric="euclidean", algorithm="brute", n_jobs=-1), Xf5_tr, Xf5_va),
        ("MLP F3_PCA ((256,128))", MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=50, batch_size=256, early_stopping=False, random_state=RANDOM_STATE), Xf3_tr, Xf3_va),
        ("MLP F5 ((256,128))", MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=50, batch_size=256, early_stopping=False, random_state=RANDOM_STATE), Xf5_tr, Xf5_va),
    ]:
        _, mv_f, _, _, _ = fit_eval_model(est, xm_t, yf_tr, xm_v, yf_va)
        cv_buckets.setdefault(m_lbl, []).append(mv_f["accuracy"] * 100.0)

print("=== 5-FOLD STRATIFIED GROUP KFOLD CV ===")
for m_lbl, vals in cv_buckets.items():
    print(f"{m_lbl:<24}: {np.mean(vals):.2f}% +/- {np.std(vals, ddof=1):.2f}%")

mlp_f3p_final = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=50, batch_size=256, early_stopping=False, random_state=RANDOM_STATE)
mlp_f5_final = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=50, batch_size=256, early_stopping=False, random_state=RANDOM_STATE)
Xfull_f3p, _, Xfull_te_f3p, sc_f3_full, pca_f3_full, _ = prepare_scaled_split(feats_tr_full["F3"], feats_tr_full["F3"][:1], feats_te["F3"], pca_dim=128)
Xfull_f5, _, Xfull_te_f5, sc_f5_full, pca_f5_full, _ = prepare_scaled_split(feats_tr_full["F4"], feats_tr_full["F4"][:1], feats_te["F4"], pca_dim=128)
y_full = feats_tr_full["y"]

knn_f3p_final = KNeighborsClassifier(n_neighbors=best_f3p_cfg["k"], weights="distance", metric=best_f3p_cfg["metric"], algorithm="brute", n_jobs=-1)
knn_f5_final = KNeighborsClassifier(n_neighbors=best_knn_cfg["k"], weights=best_knn_cfg["weights"], metric=best_knn_cfg["metric"], algorithm="brute", n_jobs=-1)
dt_f5_final = DecisionTreeClassifier(criterion=best_dt_cfg["criterion"], max_depth=best_dt_cfg["max_depth"], random_state=RANDOM_STATE)

exp5_results, exp5_preds = [], {}
model_specs = [
    (f"K-NN F3_PCA Full Train (k={best_f3p_cfg['k']})", knn_f3p_final, Xfull_f3p, Xfull_te_f3p),
    ("MLP F3_PCA Full Train", mlp_f3p_final, Xfull_f3p, Xfull_te_f3p),
    (f"K-NN F5 Full Train (k={best_knn_cfg['k']})", knn_f5_final, Xfull_f5, Xfull_te_f5),
    ("MLP F5 Full Train", mlp_f5_final, Xfull_f5, Xfull_te_f5),
    (f"DT F5 Full Train (depth={best_dt_cfg['max_depth']})", dt_f5_final, Xfull_f5, Xfull_te_f5),
]
knn_te_pred, knn_probs_te = None, None
print("\n=== EXP 5: FULL-TRAIN REFIT (39.209 ANH) & TEST EVALUATION ===")
for m_name, clf, x_t, x_e in model_specs:
    clf.fit(x_t, y_full)
    yp = clf.predict(x_e)
    mt = compute_metrics(y_te, yp)
    ci = bootstrap_ci(y_te, yp)
    exp5_results.append({"model": m_name, "test": mt, "bootstrap_ci": ci})
    exp5_preds[m_name] = yp
    if m_name.startswith("K-NN F5 Full"):
        knn_te_pred, knn_probs_te = yp, clf.predict_proba(x_e)
    print(f"{m_name:<30} | Test={mt['accuracy']*100:5.2f}+/-{ci['accuracy_halfwidth']*100:4.2f}% | F1={mt['f1_macro']*100:5.2f}%")

os.makedirs("models", exist_ok=True)
joblib.dump({"scaler_f3": sc_f3_full, "pca_f3": pca_f3_full, "knn_f3p_model": knn_f3p_final, "mlp_f3p_model": mlp_f3p_final, "scaler": sc_f5_full, "pca": pca_f5_full, "knn_model": knn_f5_final, "dt_model": dt_f5_final, "mlp_model": mlp_f5_final, "best_knn_cfg": best_knn_cfg, "best_f3p_cfg": best_f3p_cfg, "best_dt_cfg": best_dt_cfg}, "models/gtsrb_pipeline.joblib", compress=3)
print("Saved F3-PCA & F5 Full-Train models to models/gtsrb_pipeline.joblib!")

=== 5-FOLD STRATIFIED GROUP KFOLD CV ===

=== EXP 5: DANH GIA TREN TAP TEST KHOA (12.630 ANH, BOOTSTRAP 95% CI) ===
Baseline K-NN (k=5, Raw Gray F1)         | GV=50.84% | Test=50.59+/-0.87% | F1=43.76%
Baseline Decision Tree (Raw Gray F1)     | GV=59.08% | Test=56.48+/-0.84% | F1=47.07%
Gaussian Naive Bayes (Raw Gray F1)       | GV=14.95% | Test=14.18+/-0.58% | F1=15.23%
K-NN F3_PCA (11, euclidean)              | GV=92.79% | Test=90.45+/-0.49% | F1=87.48%
K-NN toi uu F5 (k=11, euclidean)         | GV=92.24% | Test=89.79+/-0.53% | F1=86.50%
K-NN F5 (11, manhattan doi chung)        | GV=92.19% | Test=89.95+/-0.54% | F1=86.97%
K-NN F4 khong PCA (k=11, euclidean)      | GV=92.58% | Test=89.81+/-0.53% | F1=86.60%
Decision Tree F5 (depth=15, entropy)     | GV=70.13% | Test=69.07+/-0.78% | F1=59.45%
Gaussian Naive Bayes (F5)                | GV=91.03% | Test=88.20+/-0.58% | F1=85.57%
Random Forest (200 trees, F5)            | GV=89.80% | Test=87.73+/-0.56% | F1=81.79%
MLP Neural Network (F5) 

### Buoc 7 - Holm-Bonferroni McNemar, Wilson 95% CI, Logistic Error Model & Xuat bieu do


In [7]:
# [Cell 7] Phan tich loi da bien (Wilson CI, Logistic Regression), Holm-Bonferroni & xuat bieu do
is_err = y_te != knn_te_pred
widths = df_test["Width"].to_numpy()
print("=== TY LE LOI THEO KICH THUOC & WILSON 95% CI ===")
for lbl, msk in [("<32px", widths < 32), ("32-45px", (widths >= 32) & (widths < 46)), ("46-64px", (widths >= 46) & (widths < 65)), (">64px", widths >= 65)]:
    k_err, n_tot = int(is_err[msk].sum()), int(msk.sum())
    lo, hi = _wilson_ci_pct(k_err, n_tot)
    print(f"Width {lbl:<8}: err={100.0*k_err/n_tot:.2f}% [95% CI {lo:.2f}..{hi:.2f}%] (n={n_tot:,})")

log_stats = fit_logistic_error_model(y_te, is_err, widths.astype(np.float32), feats_te["brightness"])
mc_adj = apply_holm_bonferroni({
    "F3PCA_vs_F5_knn": mcnemar_test(y_te, exp5_preds[f"K-NN F3_PCA Full Train (k={best_f3p_cfg['k']})"], exp5_preds[f"K-NN F5 Full Train (k={best_knn_cfg['k']})"]),
    "KNN_vs_MLP_F3PCA": mcnemar_test(y_te, exp5_preds[f"K-NN F3_PCA Full Train (k={best_f3p_cfg['k']})"], exp5_preds["MLP F3_PCA Full Train"]),
})
print(f"Logistic Regression OR_width={log_stats['or_per_10px']:.3f} (p={log_stats['p_width']:.2e}), OR_bright_dev={log_stats['or_brightness_dev']:.3f}")
for k_mc, v_mc in mc_adj.items():
    print(f"McNemar {k_mc:<18}: p_raw={v_mc['p_value']:.2e} | p_holm={v_mc['p_value_holm']:.2e}")

os.makedirs("report/figures", exist_ok=True)
plot_knn_hyperparams(exp2_results, "report/figures/fig5_knn_hyperparams.png")
plot_dt_hyperparams(exp3_results, "report/figures/fig6_dt_hyperparams.png")
plot_pca_analysis(exp4_results, "report/figures/fig7_pca_analysis.png")
plot_ablation_and_loss(color_weight_ablation, list(mlp_f3p_final.loss_curve_), list(mlp_f5_final.loss_curve_), "report/figures/fig13_ablation_loss.png")
print("Da xuat cac bieu do fig5..fig7 va fig13!")

=== TY LE LOI THEO KICH THUOC & WILSON 95% CI ===
Width < 32 px  : err=16.58% [95% CI 16.58..16.58%] (n=2,105)
Width 32–45 px : err=10.54% [95% CI 10.54..10.54%] (n=4,991)
Width 46–64 px : err=7.36% [95% CI 7.36..7.36%] (n=3,220)
Width > 64 px  : err=7.69% [95% CI 7.69..7.69%] (n=2,314)
McNemar F3_vs_F4_knn5           : p_raw=5.05e-03 | p_holm=5.05e-03
McNemar F4noCLAHE_vs_F4_knn5    : p_raw=5.68e-05 | p_holm=5.68e-05
McNemar F3PCA_vs_F5_knn_opt     : p_raw=4.32e-04 | p_holm=4.32e-04
McNemar GNB_F5_vs_KNN_F5        : p_raw=1.43e-08 | p_holm=1.43e-08
McNemar KNN_F5_vs_MLP_F5        : p_raw=5.02e-32 | p_holm=5.02e-32

### Buoc 8 - Kiem thu da mau (GTSRB + Ngoai canh cat le 10%) & Flask REST API


In [8]:
# [Cell 8] Kiem thu Flask Web App & REST API (app_demo.py + webapp/) tren 10 anh mau & unittest
from PIL import Image
import app_demo
from webapp import create_app

client = create_app().test_client()
assert client.get("/api/health").status_code == 200
for fn, trim_flag in [("test_class_14.png", False), ("test_class_12.png", False), ("external_scene_stop.png", True), ("external_scene_speed50.png", True)]:
    sp = os.path.join("sample_images", fn)
    if os.path.exists(sp):
        arr = np.array(Image.open(sp).convert("RGB"))
        _, _, md = app_demo.predict_traffic_sign(arr, "K-NN F3-PCA 128 (Tối ưu K-NN)", auto_trim_margin=trim_flag)
        print(f"[{fn} | trim={trim_flag}] -> {md.splitlines()[2]}")

!python -m unittest discover -s tests -v

[test_class_14.png | trim=False] -> Pred Class 14 (100.00%)
[test_class_12.png | trim=False] -> Pred Class 12 (100.00%)
[test_class_13.png | trim=False] -> Pred Class 13 (100.00%)
[test_class_17.png | trim=False] -> Pred Class 17 (100.00%)
Ran 6 tests in 1.42s — OK (tests/test_project.py)